# 09 — Posterior Geometry Sweep

For each of five training-set sizes (N = 500, 1000, 2000, 5000, 10000) load the
model trained in `07_sample_size_sweep.ipynb`, run SGLD to sample the posterior,
and produce a joint diagnostic panel:

1. **Posterior PCA** — 2-D projection of posterior samples; ellipsoidal shape
   indicates a regular model, any non-ellipsoidal structure signals singularities.
2. **Gen NLL vs Bayes** — where this N sits on the learning curve.
3. **KL(T_true || T_model)** — how close the learned transition matrix is to truth.

Interpreting the joint picture: if the posterior is ellipsoidal at all N and the
gen NLL tracks smoothly toward Bayes, the network is a regular model finding MLE.
Singularities would appear as non-Gaussian posterior shapes and potentially
plateau-like NLL behaviour — a signal that circuit tracing or operator-spectrum
analysis would be warranted.

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
# ── Resolve project root ──────────────────────────────────────────────────────
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt = cwd / "projects" / "markov-chain-learning"
    if (alt / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt
if project_root is None:
    raise RuntimeError("Could not locate project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer
from torch_bdn.bn import BayesianNet
from torch_bdn.sampling import Sampler

DATA_DIR = project_root / "experiments" / "single-chain" / "data"
print("Data dir:", DATA_DIR.resolve())

## Load sweep records and reference DGP

In [ ]:
records = torch.load(DATA_DIR / "sample_size_sweep.pt", weights_only=False)

# Verify model weights were saved (requires re-running 07 after the patch)
if "model_state_dict" not in records[0]:
    raise RuntimeError(
        "model_state_dict not found in sweep records.\n"
        "Re-run 07_sample_size_sweep.ipynb first — it now saves model weights."
    )

record_by_N = {r["N"]: r for r in records}
print("Available N:", sorted(record_by_N.keys()))

In [ ]:
data = torch.load(DATA_DIR / "sequences.pt", weights_only=False)
data_cfg = data.get("config", {})
Ts = data["Ts"]
pis = data["pis"]
chain_to_sample = data["chain_to_sample"]

VOCAB = int(data_cfg.get("n_states", Ts.shape[-1]))
MAX_LEN = int(data_cfg.get("L", data_cfg.get("seq_len", 32)))
PAD_ID = int(data_cfg.get("pad_id", -1))
D_MODEL = VOCAB * 3

if Ts.shape[0] == 1:
    T_reference = Ts[0]
    pi_reference = pis[0]
else:
    chain_weights = torch.bincount(chain_to_sample, minlength=Ts.shape[0]).float()
    chain_weights /= chain_weights.sum()
    T_reference = (chain_weights[:, None, None] * Ts).sum(0)
    pi_reference = (chain_weights[:, None] * pis).sum(0)

print(f"VOCAB={VOCAB}  MAX_LEN={MAX_LEN}  PAD_ID={PAD_ID}")

## Build fixed test set (same as 07)

In [ ]:
from markov_chain import sample_sequences

N_TEST = 10_000
test_seqs_np = sample_sequences(
    T_reference.numpy(),
    n=N_TEST,
    seq_len=MAX_LEN,
    prior=pi_reference.numpy(),
    rng=0,
)
test_seqs = torch.from_numpy(test_seqs_np).long()
test_x_in = test_seqs[:, :-1]
test_x_tgt = test_seqs[:, 1:]

rng_test = np.random.default_rng(0)
test_lengths = rng_test.integers(low=2, high=MAX_LEN + 1, size=N_TEST)
test_valid = np.arange(MAX_LEN - 1)[None, :] < (test_lengths[:, None] - 1)

log_T_ref = torch.log(T_reference.clamp_min(1e-12))
bayes_nll = float(-log_T_ref[test_x_in, test_x_tgt].float().numpy()[test_valid].mean())
print(f"Bayes NLL: {bayes_nll:.6f}")

## SGLD configuration and helpers

In [ ]:
# SGLD hyperparameters — keep consistent with 05_analysis.ipynb
SGLD_LR = 0.2
SGLD_WARMUP = 1100
SGLD_NUM_SAMPLES = 2000
SGLD_THIN = 20
SGLD_EXTRA_BURNIN = 100  # additional samples dropped after warmup
SIGMA_PRIOR = 4.0

TARGET_NS = [500, 1_000, 2_000, 5_000, 10_000]


def make_loss_fn(pad_id, vocab):
    def loss_fn(logits, targets):
        ce = F.cross_entropy(
            logits.view(-1, vocab),
            targets.view(-1),
            reduction="none",
            ignore_index=pad_id,
        )
        ce = ce.view(logits.shape[:-1])
        mask = (targets != pad_id).float()
        return (ce * mask).sum() / mask.sum()

    return loss_fn


def make_prior_logp(sigma):
    log_2pi = math.log(2 * math.pi)

    def prior_logp(params):
        num_params = params.numel()
        quad = params.pow(2).sum()
        return -0.5 * num_params * log_2pi - 0.5 * quad / (sigma**2)

    return prior_logp


def run_sgld_for_record(record):
    """Load trained model from sweep record and sample its posterior with SGLD."""
    N = record["N"]
    # Reconstruct training sequences (same RNG as sweep notebook)
    SEED = 42
    seqs_np = sample_sequences(
        T_reference.numpy(),
        n=N,
        seq_len=MAX_LEN,
        prior=pi_reference.numpy(),
        rng=SEED + N,
    )
    seqs = torch.from_numpy(seqs_np).long()
    x_data = seqs[:, :-1].clone()
    x_data[x_data == PAD_ID] = 0
    y_data = seqs[:, 1:]

    # Load model from saved weights
    model = MarkovTransformer(vocab_size=VOCAB, d_model=D_MODEL, max_len=MAX_LEN)
    model.load_state_dict(record["model_state_dict"])
    model.eval()

    loss_fn = make_loss_fn(PAD_ID, VOCAB)
    prior_logp = make_prior_logp(SIGMA_PRIOR)
    bn = BayesianNet(model, loss_fn, prior_logp)
    sampler = Sampler(bn, x_data, y_data)

    samples = sampler.sample(
        num_samples=SGLD_NUM_SAMPLES,
        backend="sgld",
        lr=SGLD_LR,
        warmup=SGLD_WARMUP,
        thin=SGLD_THIN,
    )

    chain_full = torch.stack(samples["parameters"]).cpu().numpy()  # (n_samples, D)
    chain = chain_full[SGLD_EXTRA_BURNIN:, :]
    print(f"  N={N:>6}  posterior chain shape: {chain.shape}")
    return chain

## Run SGLD for each target N

This cell is the slow step — ~5 min per N on CPU/MPS.

In [ ]:
posterior_chains = {}  # N -> np.ndarray (n_samples, D)

for N in TARGET_NS:
    if N not in record_by_N:
        print(f"N={N} not in sweep records, skipping")
        continue
    print(f"Sampling posterior for N={N}...")
    posterior_chains[N] = run_sgld_for_record(record_by_N[N])

print("\nDone.")

## Joint diagnostic panels

Five columns (one per N), three rows:
- **Row 1** — Posterior in PC1/PC2 space.  An ellipsoidal cloud = regular model.
- **Row 2** — Per-sample log-posterior trace (SGLD mixing check).
- **Row 3** — Posterior-mean predicted vs true transition matrix (heatmap diff).

In [ ]:
fig, axes = plt.subplots(3, len(TARGET_NS), figsize=(4 * len(TARGET_NS), 11))

for col, N in enumerate(TARGET_NS):
    if N not in posterior_chains:
        continue

    chain = posterior_chains[N]  # (n_samples, D)
    rec = record_by_N[N]

    # ── Row 0: PCA scatter ────────────────────────────────────────────────
    ax = axes[0, col]
    pca = PCA(n_components=2)
    proj = pca.fit_transform(chain)  # (n_samples, 2)
    var_explained = pca.explained_variance_ratio_

    ax.scatter(
        proj[:, 0], proj[:, 1], alpha=0.15, s=4, color="steelblue", rasterized=True
    )
    ax.set_title(f"N={N:,}", fontsize=10, fontweight="bold")
    ax.set_xlabel(f"PC1 ({var_explained[0]:.1%})", fontsize=8)
    ax.set_ylabel(f"PC2 ({var_explained[1]:.1%})", fontsize=8)
    if col == 0:
        ax.set_ylabel(f"PC2 ({var_explained[1]:.1%})\nPosterior PCA", fontsize=8)

    # Annotate with gen NLL and KL to truth
    ax.annotate(
        f"gen NLL={rec['gen_nll']:.4f}\nKL→true={rec['kl_to_true']:.4f}",
        xy=(0.03, 0.97),
        xycoords="axes fraction",
        va="top",
        ha="left",
        fontsize=7,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7),
    )

    # ── Row 1: Log-posterior trace ────────────────────────────────────────
    ax = axes[1, col]
    # Recompute log-posterior norm as a proxy for mixing: trace of ||θ - mean||²
    mean_params = chain.mean(axis=0)
    dist_to_mean = np.linalg.norm(chain - mean_params[None, :], axis=1)
    ax.plot(dist_to_mean, lw=0.6, alpha=0.8, color="darkorange")
    ax.axhline(dist_to_mean.mean(), color="gray", lw=0.8, linestyle="--")
    ax.set_xlabel("Sample index", fontsize=8)
    if col == 0:
        ax.set_ylabel("||θ − mean||", fontsize=8)
    ax.set_title(f"σ={dist_to_mean.std():.3f}", fontsize=8)

    # ── Row 2: T_true − T_model heatmap ───────────────────────────────────
    ax = axes[2, col]
    T_model = torch.tensor(rec["T_model"], dtype=torch.float32)
    T_diff = (T_reference - T_model).numpy()
    vmax = max(abs(T_diff).max(), 0.01)
    im = ax.imshow(T_diff, cmap="RdBu", vmin=-vmax, vmax=vmax, aspect="auto")
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_xlabel("Next state j", fontsize=8)
    if col == 0:
        ax.set_ylabel("Current state i\nT_true − T_model", fontsize=8)
    ax.set_title(f"max|Δ|={abs(T_diff).max():.3f}", fontsize=8)

fig.suptitle(
    "Posterior geometry across training-set sizes",
    fontsize=13,
    fontweight="bold",
    y=1.01,
)
plt.tight_layout()
plt.savefig(DATA_DIR / "posterior_sweep.pdf", bbox_inches="tight")
plt.show()
print("Saved:", DATA_DIR / "posterior_sweep.pdf")

## Posterior volume across N

Plot the log-determinant of the posterior covariance (in PCA space, top-K components)
as a function of N.  For a regular model this should decrease like $-\log N$ (Fisher
information accumulates linearly).  Any deviation from linearity on a log-log plot
would suggest an effective RLCT $\lambda \neq d/2$.

In [ ]:
K_PCA = 10  # top-K PCs to use for volume estimate

ns_computed = []
log_det_covs = []

for N in TARGET_NS:
    if N not in posterior_chains:
        continue
    chain = posterior_chains[N]
    pca = PCA(n_components=K_PCA)
    pca.fit(chain)
    # log-det of covariance in top-K PCA space ≈ sum of log eigenvalues
    log_det = float(np.sum(np.log(pca.explained_variance_ + 1e-30)))
    ns_computed.append(N)
    log_det_covs.append(log_det)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    np.log(ns_computed),
    log_det_covs,
    "o-",
    color="steelblue",
    label="log det Σ (top-K PCA)",
)

# Reference slope: regular model => log det Σ ∝ -K * log N
if len(ns_computed) >= 2:
    x0, x1 = np.log(ns_computed[0]), np.log(ns_computed[-1])
    y0 = log_det_covs[0]
    ref_slope = -K_PCA
    ax.plot(
        [x0, x1],
        [y0, y0 + ref_slope * (x1 - x0)],
        "--",
        color="gray",
        lw=1,
        label=f"Regular model slope (−{K_PCA})",
    )

ax.set_xlabel("log N", fontsize=11)
ax.set_ylabel(f"log det Σ (top-{K_PCA} PCs)", fontsize=11)
ax.set_title("Posterior volume vs sample size", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(DATA_DIR / "posterior_volume.pdf", bbox_inches="tight")
plt.show()
print("Saved:", DATA_DIR / "posterior_volume.pdf")

## Summary

**What to look for:**

| Observation | Interpretation |
|---|---|
| Row-0 ellipses tighten uniformly as N grows | Regular model — posterior concentrates as expected |
| Any non-ellipsoidal / multi-modal cloud | Singular geometry — the model has degenerate parameters |
| Volume plot slope ≈ −K | RLCT ≈ d/2 — confirms regularity |
| Volume plot slope < −K (faster decay) | Effective RLCT < d/2 — singularity helping generalisation |
| gen NLL stays well above Bayes at large N | Model stuck — warrants circuit tracing / operator spectrum analysis |

If the posterior is fully ellipsoidal and volume decays at the regular rate, the
network is doing standard MLE convergence and circuit tracing is unlikely to reveal
non-trivial structure beyond the learned transition-matrix lookup.